# 🐻 H.U.G.H. Personality LoRA Training — v9 Omniversal
**VERBOSE LOGGING | 279 pairs | 5 epochs | LoRA r=32 | QLoRA 4-bit**


In [ ]:
import sys
print('=== CELL 1: ENVIRONMENT SETUP ===')
sys.stdout.flush()

# Install WITHOUT -q so failures are visible
print('>>> pip install transformers peft datasets accelerate bitsandbytes...')
sys.stdout.flush()
!pip install 'transformers>=4.45.0' 'peft>=0.13.0' datasets accelerate bitsandbytes 2>&1 |  
print('>>> pip install sentencepiece tqdm huggingface_hub...')
sys.stdout.flush()
!pip install sentencepiece tqdm 'huggingface_hub>=0.24' 2>&1 | tail -5

print('>>> Importing torch...')
sys.stdout.flush()
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {props.total_memory/1e9:.1f}GB')
sys.stdout.flush()

print('>>> HF login...')
sys.stdout.flush()
from huggingface_hub import login
login(token='os.environ["HF_TOKEN"]')
print('✅ HF login OK')
sys.stdout.flush()
print('=== CELL 1 COMPLETE ===')
sys.stdout.flush()


In [ ]:
import os, glob, subprocess

# Bulletproof dataset finder - searches EVERYWHERE then downloads as fallback
dataset_path = None
filename = "hugh_personality_training_v2.jsonl"

# 1. Search all of /kaggle/input recursively
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f == filename:
            dataset_path = os.path.join(root, f)
            break
    if dataset_path:
        break

# 2. Check /kaggle/working
if not dataset_path and os.path.exists(f"/kaggle/working/{filename}"):
    dataset_path = f"/kaggle/working/{filename}"

# 3. Glob fallback - find ANY .jsonl file
if not dataset_path:
    jsonl_files = glob.glob("/kaggle/input/**/*.jsonl", recursive=True)
    if jsonl_files:
        dataset_path = jsonl_files[0]
        print(f"Found alternate JSONL: {dataset_path}")

# 4. Nuclear option - download via Kaggle API
if not dataset_path:
    print("File not in /kaggle/input — downloading via API...")
    subprocess.run(["pip", "install", "-q", "kaggle"], check=True)
    subprocess.run([
        "kaggle", "datasets", "download", 
        "grizzlymedicine/hugh-sovereign-v3",
        "-p", "/kaggle/working/", "--unzip"
    ], check=True)
    if os.path.exists(f"/kaggle/working/{filename}"):
        dataset_path = f"/kaggle/working/{filename}"

if not dataset_path:
    # 5. Absolute last resort - list everything
    print("STILL not found. Full /kaggle/input tree:")
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            print(f"  {os.path.join(root, f)}")
    raise FileNotFoundError(f"Cannot find {filename} anywhere!")

print(f"✅ Dataset: {dataset_path}")
print(f"   Size: {os.path.getsize(dataset_path) / 1e6:.2f} MB")

# Count pairs
with open(dataset_path) as f:
    lines = f.readlines()
print(f"   Pairs: {len(lines)}")


# Load the dataset into memory
import json as _json
pairs = []
with open(dataset_path) as f:
    for line in f:
        line = line.strip()
        if line:
            pairs.append(_json.loads(line))

print(f"   Loaded {len(pairs)} training pairs ✅")


In [ ]:
# ═══════════════════════════════════════════════════════
# CONFIGURATION — Adjust these as needed
# ═══════════════════════════════════════════════════════

MODEL_ID = "DavidAU/LFM2.5-1.2B-Thinking-Claude-4.6-Opus-Heretic-Uncensored-DISTILL"
FALLBACK_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# LoRA config — stronger signal for personality imprint
LORA_R = 32          # Rank (was 16, doubled for richer dataset)
LORA_ALPHA = 64      # Alpha = 2x rank
LORA_DROPOUT = 0.05

# Training config
EPOCHS = 10           # Was 3, increased for 279 pairs
LEARNING_RATE = 1e-4 # Slightly lower for stability
BATCH_SIZE = 2       # Per-device (T4 has 16GB VRAM)
GRAD_ACCUM = 8       # Effective batch = 16
MAX_LENGTH = 2048    # Max sequence length
WARMUP_RATIO = 0.1

# QLoRA — 4-bit quantization to fit on T4
USE_QLORA = True

# Output
OUTPUT_DIR = "/kaggle/working/hugh_personality_lora_v2"
HF_REPO = "oldmangrizzz/hugh-personality-lora-v2"

print("Configuration loaded ✅")
print(f"  Model: {MODEL_ID}")
print(f"  LoRA: r={LORA_R}, α={LORA_ALPHA}")
print(f"  Training: {EPOCHS} epochs, lr={LEARNING_RATE}, effective_batch={BATCH_SIZE * GRAD_ACCUM}")


In [ ]:
import sys
print('=== CELL 4: MODEL LOAD ===')
sys.stdout.flush()

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = MODEL_ID
try:
    print(f"Loading tokenizer: {model_id}")
    sys.stdout.flush()
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, token="os.environ["HF_TOKEN"]")
    print(f"  ✅ Tokenizer loaded")
    sys.stdout.flush()
except Exception as e:
    print(f"  ❌ Primary tokenizer failed: {e}")
    sys.stdout.flush()
    model_id = FALLBACK_MODEL
    print(f"  Falling back to {model_id}")
    sys.stdout.flush()
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, token="os.environ["HF_TOKEN"]")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model_kwargs = {"trust_remote_code": True}
if USE_QLORA and torch.cuda.is_available():
    print("Using 4-bit QLoRA quantization")
    sys.stdout.flush()
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model_kwargs["quantization_config"] = bnb_config
elif torch.cuda.is_available():
    model_kwargs["torch_dtype"] = torch.bfloat16

print(f"Loading model: {model_id}")
sys.stdout.flush()
model = AutoModelForCausalLM.from_pretrained(model_id, token="os.environ["HF_TOKEN"]", **model_kwargs)
print(f"  ✅ Model loaded: {model.config.model_type}")
sys.stdout.flush()
print(f"  Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")
sys.stdout.flush()

print('=== CELL 4 COMPLETE ===')
sys.stdout.flush()


In [ ]:
# ═══════════════════════════════════════════════════════
# APPLY LoRA ADAPTER
# ═══════════════════════════════════════════════════════
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

if USE_QLORA:
    model = prepare_model_for_kbit_training(model)

# Auto-detect target modules
print("Scanning model for target modules...")
linear_modules = set()
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        # Get just the last part of the name
        short_name = name.split('.')[-1]
        linear_modules.add(short_name)
print(f"  Found linear modules: {sorted(linear_modules)}")

# Prefer attention projections; fall back to all linear
preferred = ["q_proj", "k_proj", "v_proj", "o_proj"]
available = [m for m in preferred if m in linear_modules]
if not available:
    # Try GPT-style names
    preferred_gpt = ["c_attn", "c_proj", "c_fc"]
    available = [m for m in preferred_gpt if m in linear_modules]
if not available:
    available = list(linear_modules)

print(f"  Target modules: {available}")

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=available,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# ═══════════════════════════════════════════════════════
# PREPARE DATASET
# ═══════════════════════════════════════════════════════
from datasets import Dataset

def format_conversation(entry):
    """Format a conversation entry for training."""
    messages = entry.get("messages", [])
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False)
    except Exception:
        parts = []
        for msg in messages:
            role = msg["role"]
            content = msg["content"]
            if role == "system":
                parts.append(f"<|system|>\n{content}")
            elif role == "user":
                parts.append(f"<|user|>\n{content}")
            elif role == "assistant":
                parts.append(f"<|assistant|>\n{content}")
        return "\n".join(parts)

# Format all pairs
texts = [format_conversation(p) for p in pairs]
dataset = Dataset.from_dict({"text": texts})

# Tokenize
def tokenize_fn(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

print(f"Tokenizing {len(dataset)} examples (max_length={MAX_LENGTH})...")
tokenized_dataset = dataset.map(tokenize_fn, remove_columns=["text"], num_proc=2)

# Split
split = tokenized_dataset.train_test_split(test_size=0.05, seed=42)
print(f"  Train: {len(split['train'])}, Eval: {len(split['test'])}")

# Check token lengths
sample_lens = [sum(1 for t in ex['input_ids'] if t != tokenizer.pad_token_id) for ex in split['train'].select(range(min(50, len(split['train']))))]
print(f"  Sample token lengths: min={min(sample_lens)}, avg={sum(sample_lens)//len(sample_lens)}, max={max(sample_lens)}")


In [ ]:
# ═══════════════════════════════════════════════════════
# TRAIN
# ═══════════════════════════════════════════════════════
import time
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    logging_steps=1,
    logging_first_step=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=torch.cuda.is_available(),
    report_to="none",
    dataloader_pin_memory=True,
    gradient_checkpointing=True,  # Save VRAM
    optim="paged_adamw_8bit" if USE_QLORA else "adamw_torch",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    data_collator=data_collator,
)

print(f"{'='*60}")
print(f"🐻 H.U.G.H. PERSONALITY TRAINING — COMMENCING")
print(f"{'='*60}")
print(f"  Pairs: {len(pairs)}")
print(f"  Epochs: {EPOCHS}")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Steps/epoch: ~{len(split['train']) // (BATCH_SIZE * GRAD_ACCUM)}")
print(f"  Total steps: ~{len(split['train']) // (BATCH_SIZE * GRAD_ACCUM) * EPOCHS}")
print(f"{'='*60}")

start_time = time.time()
trainer.train()
elapsed = time.time() - start_time
print(f"\n✅ Training complete in {elapsed/60:.1f} minutes")


In [ ]:
# ═══════════════════════════════════════════════════════
# SAVE ADAPTER + INFERENCE TEST
# ═══════════════════════════════════════════════════════
import json as json_mod

adapter_dir = os.path.join(OUTPUT_DIR, "adapter")
os.makedirs(adapter_dir, exist_ok=True)

# Save adapter
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

# Save training metadata
meta = {
    "base_model": model_id,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "training_pairs": len(pairs),
    "training_time_minutes": round(elapsed / 60, 1),
    "persona": "H.U.G.H. (Hyper-Unified Guardian and Harbormaster)",
    "version": "v2-omniversal",
    "domains": "coding, pop-culture, science, philosophy, ems, security, music, comics, crossover",
    "description": "Personality LoRA v2 — 279 pairs across 12+ domains with McGregor warmth × Mills lethality × Lad energy",
}
with open(os.path.join(adapter_dir, "hugh_training_meta.json"), "w") as f:
    json_mod.dump(meta, f, indent=2)

print(f"✅ Adapter saved to {adapter_dir}/")
print(f"   Files: {os.listdir(adapter_dir)}")

# Inference test
print(f"\n{'='*60}")
print("🧪 INFERENCE TEST")
print(f"{'='*60}")

model.eval()
test_prompts = [
    "What wisdom do you seek today?",
    "Who said 'The Almighty tells me he can get me out of this mess' in Braveheart?",
    "Explain quantum entanglement using a movie analogy.",
    "Someone's trying to social engineer into our network. What do we do?",
    "What's your favorite album?",
    "I just mass-deleted the production database.",
]

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": pairs[0]['messages'][0]['content'][:500]},  # Use v2 system prompt
        {"role": "user", "content": prompt}
    ]
    try:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except:
        text = f"<|system|>\nYou are H.U.G.H.\n<|user|>\n{prompt}\n<|assistant|>\n"
    
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=200, temperature=0.7,
            do_sample=True, top_p=0.9, repetition_penalty=1.15
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n  Q: {prompt}")
    print(f"  A: {response[:300]}")
    print(f"  ---")


In [ ]:
# ═══════════════════════════════════════════════════════
# UPLOAD TO HUGGINGFACE (optional)
# ═══════════════════════════════════════════════════════
from huggingface_hub import HfApi, login

# Set your HF token
HF_TOKEN = "os.environ["HF_TOKEN"]"  # Replace with your actual token

try:
    login(token=HF_TOKEN)
    api = HfApi()
    
    # Create repo if needed
    try:
        api.create_repo(repo_id=HF_REPO, repo_type="model", exist_ok=True)
    except:
        pass
    
    # Upload adapter
    api.upload_folder(
        folder_path=adapter_dir,
        repo_id=HF_REPO,
        repo_type="model",
        commit_message=f"Hugh personality LoRA v2 — {len(pairs)} pairs, {EPOCHS} epochs"
    )
    print(f"✅ Uploaded to https://huggingface.co/{HF_REPO}")
except Exception as e:
    print(f"❌ HF upload failed: {e}")
    print(f"   Adapter is saved locally at {adapter_dir}/")
    print(f"   Download from Kaggle output and upload manually")


In [ ]:
# ═══════════════════════════════════════════════════════
# TRAINING SUMMARY
# ═══════════════════════════════════════════════════════
print(f"""
╔══════════════════════════════════════════════════════════╗
║  🐻 H.U.G.H. PERSONALITY TRAINING COMPLETE             ║
╠══════════════════════════════════════════════════════════╣
║  Model:    {model_id:<46}║
║  Pairs:    {len(pairs):<46}║
║  Epochs:   {EPOCHS:<46}║
║  LoRA:     r={LORA_R}, α={LORA_ALPHA}{' '*(41-len(f'r={LORA_R}, α={LORA_ALPHA}'))}║
║  Time:     {elapsed/60:.1f} minutes{' '*(40-len(f'{elapsed/60:.1f} minutes'))}║
║  Adapter:  {adapter_dir:<46}║
╚══════════════════════════════════════════════════════════╝

Next steps:
  1. Download adapter from Kaggle output (or check HuggingFace)
  2. SCP to CT102: scp -r adapter/ root@192.168.7.102:/opt/hugh/models/personality_lora_v2/
  3. Update llama.cpp to load LoRA adapter
  4. Restart Hugh runtime

The Workshop awaits. 🐻
""")
